In [1]:
# inference using deberta-v3-base-nli

In [2]:
import os

ROOT = "/kaggle/input/kdsh26-deberta-v3-base-fine-tune-model"

def find_model_dir(root):
    for dirpath, dirnames, filenames in os.walk(root):
        if "config.json" in filenames and (
            "model.safetensors" in filenames or "pytorch_model.bin" in filenames
        ):
            print("Using model dir:", dirpath)
            return dirpath
    raise RuntimeError("No directory with config.json + model.safetensors/pytorch_model.bin found")

MODEL_DIR = find_model_dir(ROOT)


Using model dir: /kaggle/input/kdsh26-deberta-v3-base-fine-tune-model/kaggle/working/deberta-v3-base-castaways-monte


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# base tokenizer (same as training)
BASE_MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

# fine-tuned model weights from the detected directory
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print("id2label:", model.config.id2label)


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
2026-01-10 10:06:00.081885: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768039560.503308      44 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768039560.634629      44 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register fa

id2label: {0: 'consistent', 1: 'contradict'}


In [4]:
import json
from pathlib import Path
import pandas as pd

TEST_PATH = "/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/test.csv"
MC_CONS_PATH = "/kaggle/input/kdsh26-jsonl-file-characters/monte_cristo/monte_cristo_constraints_updated.jsonl"
CA_CONS_PATH = "/kaggle/input/kdsh26-jsonl-file-characters/castaways/castaways_constraints_filled.jsonl"

test_df = pd.read_csv(TEST_PATH)

def load_constraints(path):
    mapping = {}
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            key = (obj["book_name"], obj["character"])
            mapping[key] = obj.get("constraints", [])
    return mapping

mc_constraints = load_constraints(MC_CONS_PATH)
ca_constraints = load_constraints(CA_CONS_PATH)
constraints = {**mc_constraints, **ca_constraints}

def constraint_to_sentence(book, char, c):
    dim = c["dimension"]; val = c["value"]
    if dim == "health_state":
        return f"In {book}, {char} is {val}."
    elif dim == "family_role":
        return f"In {book}, {char} has family role: {val}."
    elif dim == "role":
        return f"In {book}, {char} is described as {val}."
    elif dim == "geographic_expertise":
        return f"{char} is familiar with {val}."
    elif dim == "criminal_history":
        return f"{char} has criminal history: {val}."
    else:
        return f"{dim}: {val}."

def build_context(book, char, max_cons=6):
    cons = constraints.get((book, char), [])
    if not cons:
        return ""
    sents = [constraint_to_sentence(book, char, c) for c in cons[:max_cons]]
    return " ".join(sents)

test_df["context"] = test_df.apply(
    lambda r: build_context(r["book_name"], r["char"]), axis=1
)


In [5]:
def predict_batch(texts, batch_size=32):
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            logits = model(**enc).logits
            preds = logits.argmax(dim=-1).cpu().tolist()
        all_preds.extend(preds)
    return all_preds


In [6]:
# Build the same text template used during training
inference_texts = [
    f"Premise: {ctx}\nHypothesis: {claim}"
    for ctx, claim in zip(test_df["context"].fillna(""), test_df["content"])
]

pred_ids = predict_batch(inference_texts, batch_size=32)

# Map ids back to string labels using the config
id2label = {int(k): v for k, v in model.config.id2label.items()}
test_df["label"] = [id2label[i] for i in pred_ids]

submission = test_df[["id", "label"]]
submission.to_csv("submission.csv", index=False)
submission.head()

,id,label
0,95,contradict
1,136,consistent
2,59,consistent
3,60,consistent
4,124,contradict
